In [1]:
import agents

In [2]:
#import llm
from langchain_openai import OpenAIEmbeddings

In [3]:
MODEL_EMBEDDING = "text-embedding-nomic-embed-text-v1.5"
MODEL_LLM = "google/gemma-3-1b"
BASE_URL_SERVER = "http://127.0.0.1:1234/v1"
API_KEY_SERVER = "lm-studio"

## Criar ChromaDB (ou carrega o existente)

In [4]:
embeddings = OpenAIEmbeddings(
        model=MODEL_EMBEDDING,
        base_url=BASE_URL_SERVER,
        api_key=API_KEY_SERVER,
        chunk_size=1,
        check_embedding_ctx_length=False
    )

In [5]:
agente_1 = agents.CreateVectorDB()

In [6]:
agente_1.set_collection_name("teste_base")
agente_1.set_persist_directory("./teste")
agente_1.set_embedding_function(embeddings)

In [7]:
agente_1.create_chromadb()

## Criar chunks a partir de arquivo PDF

In [10]:
chunks_pdf = agents.CreateChunks()

In [11]:
chunks_pdf.load_file_pdf(
    path_file="Manual de Pesquisa Patrimonial TRT 16 p 1-10.pdf",
    metadados_categoria_genero="investigação patrimonial",
    metadados_categoria_especie="patrimônio")


chunks = chunks_pdf.chunks(
    chunk_size=850,
    chunk_overlap=150,
    database="info_docs"
)

In [12]:
agente_1.add_chunk_to_vectorstore(
    chunks=chunks,
    batch_size=1
)

100%|██████████████████████████████████████████████████████████████████████████████████| 20/20 [00:47<00:00,  2.39s/it]


### retorna a **vectorstore - memória** para consulta

In [8]:
vectorstore = agente_1.return_vectorstore()

## Cria o agente através da classe `Agent`

In [9]:
investigador=agents.Agent()
investigador.set_agent_name("inspetorA")
investigador.set_agent_persona("Você é um experiente investigador de polícia")
investigador.set_agent_skills("Você possui habilidades robustas em analisar dados, pensamento crítico, analítico, com boa fluência e vocabulário técnico e jurídico")
investigador.set_agent_task("Responda a pergunta do usuário realizando a consulta no banco de conhecimento, redigindo uma resposta o mais completa possível. Não invente dados ou informações que não estejam na base de dados")
investigador.set_agent_knowledge(vectorstore)
investigador.setup_llm_and_chain(model_llm=MODEL_LLM, base_url=BASE_URL_SERVER, openai_api_key=API_KEY_SERVER)
investigador.create_memory(embeddings=embeddings)

### Realiza interação com o agente, através de `user_prompt`

In [15]:
investigador.run_agent(user_prompt="o que as fontes falam sobre CCS? Qual o seu significado?", use_memory=False)

### Obtém a resposta do agente...

In [16]:
out = investigador.get_agent_answer()

In [17]:
print(out["answer"])

O sistema consultará a base de dados da Receita Federal para identificar os relacionamentos bancários constantes na base de dados do Cadastros de Clientes do Sistema Financeiro Nacional (CCS). O CCS é um registro de clientes de bancos e instituições financeiras, que serve como uma espécie de "lista" de contas.

Em resumo, o CCS é a base de dados utilizada para verificar se o CPF/CNPJ do autor/exequente da ação está associado a alguma conta bancária.


In [18]:
print(out["memory_docs"])

[]


### ...e adiciona a resposta à **memória** do agente...

In [13]:
investigador.add_info_to_memory()

<class 'list'>
[Document(metadata={'Memória de Agente': 'o que as fontes falam sobre CCS? Qual o seu significado?'}, page_content='o que as fontes falam sobre CCS? Qual o seu significado?\nO sistema consultará a base de dados da Receita Federal para identificar os relacionamentos bancários constantes no CCS. O CCS é um sistema que armazena informações sobre contas bancárias, incluindo números de agências e contas.\n\nEm resumo, o CCS (Cadastro de Clientes do Sistema Financeiro Nacional) é uma ferramenta utilizada pelo Banco Central do Brasil para registrar informações sobre as contas bancárias dos cidadãos e empresas.  É fundamental para a identificação de relacionamentos bancários entre os usuários.')]


100%|████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:02<00:00,  2.85s/it]


In [14]:
for i, text in enumerate(out["top_text_rag"]):
    print(f"recuperação -> {i}\n")
    print(text)
    

recuperação -> 0

page_content='• Tipo/natureza de ação; 
• CPF/CNPJ do autor/exequente da ação; 
• Nome do autor/exequente da ação; e 
• CPF/CPNJ do réu/executado. 
No que se refere à inclusão dos réus/executados, o sistema consultará a base de dados da  
Receita Federal e informará o nome apontando os relacionamentos bancários constantes na  
base de dados do Cadastros de Clientes do Sistema Financeiro Nacional do Banco Central (CCS). 
• Caso haja mais de um réu/executado, após a indicação do primeiro CPF/CNPJ, o usuário 
deve clicar no ícone <+> para inserir os demais. Automaticamente, o sistema consultará 
o CCS e indicará os relacionamentos bancários.' metadata={'source': 'Manual de Pesquisa Patrimonial TRT 16 p 1-10.pdf', 'page_label': '9', 'page': 8, 'total_pages': 10, 'investigação patrimonial': 'patrimônio', 'creator': 'PDFium', 'producer': 'PDFium', 'creationdate': 'D:20260523185710'}
recuperação -> 1

page_content='Manual de Ferramentas de Pesquisas Eletrônicas do TRT16-MA  

In [15]:
print(out["user_prompt"])

o que as fontes falam sobre CCS? Qual o seu significado?


### ...acessando a memória do agente...

In [10]:
investigador.access_memory(user_prompt="Busque na memória informações sobre investigação patrimonial", retriever_type="mmr")

In [11]:
memory = investigador.get_agent_memory()

In [12]:
print(memory["answer_memory"])
print(memory["docs_memory"])

Com base no contexto fornecido, o CCS (Cadastro de Clientes do Sistema Financeiro Nacional) é uma ferramenta crucial para o Banco Central do Brasil, utilizada para registrar informações detalhadas sobre as contas bancárias dos cidadãos e empresas. Sua principal função é identificar e monitorar relacionamentos bancários entre os usuários, auxiliando na investigação patrimonial e no combate à lavagem de dinheiro.

**Análise Detalhada:**

*   **Função Principal:** O CCS serve como um registro centralizado das informações sobre contas bancárias, permitindo que o Banco Central tenha uma visão abrangente dos relacionamentos entre clientes e instituições financeiras.
*   **Informações Registradas:**  O sistema armazena dados como números de agências bancárias, nomes de contas, detalhes de transações e outras informações relevantes para a identificação de padrões e atividades suspeitas.
*   **Importância para Investigação Patrimonial:** A capacidade de rastrear relacionamentos bancários é fund